# Create wav2vec 2.0 embedding

In [ ]:
import pandas as pd
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model

In [7]:
train_df = pd.read_pickle("train_set.pkl")
test_df = pd.read_pickle("test_set.pkl")

print(test_df)

                                           signal_data language speaker  \
104  [-0.0234375, -0.015625, -0.0078125, -0.0078125...       SI       C   
218  [-0.14410400390625, -0.144775390625, -0.145050...       SA       E   
64   [-0.0078125, -0.0078125, -0.0078125, 0.0, -0.0...       IT       D   
6    [-3.0517578125e-05, 0.0, 0.0, -3.0517578125e-0...       PO       A   
127  [-0.000152587890625, 0.0001220703125, 0.000152...       FR       B   
164  [-3.0517578125e-05, 0.0, 3.0517578125e-05, -6....       FR       F   
0    [0.0, 3.0517578125e-05, -6.103515625e-05, -3.0...       PO       A   
59   [-0.015625, -0.0078125, -0.0078125, -0.0078125...       IT       D   
4    [-3.0517578125e-05, 3.0517578125e-05, 0.0, 0.0...       PO       A   
38   [-0.03369140625, -0.039459228515625, -0.032470...       IT       B   
71   [0.00067138671875, 0.001068115234375, 0.002471...       IT       E   
162  [-3.0517578125e-05, 0.0, 0.0, 0.0, 3.051757812...       FR       G   
186  [9.1552734375e-05, -

In [ ]:

model_name = "facebook/wav2vec2-xls-r-300m" # Using the multilingual model
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

# Set device to Apple Silicon GPU (mps) if available, otherwise fallback to cpu
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS!")
else:
    device = torch.device("cpu")
    print("MPS not found, using CPU.")

model.to(device)
model.eval()

# (The rest of the extraction function remains exactly the same)

/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# 1. Load the Processor and Model
model_name = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

# Move model to GPU if available for faster processing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set to evaluation mode

# 2. Define the extraction function
def get_w2v_embedding(waveform, sampling_rate=16000):
    # Convert list to numpy array if it isn't already
    if isinstance(waveform, list):
        waveform = np.array(waveform)
        
    # Process the waveform (padding, normalization)
    inputs = processor(
        waveform, 
        sampling_rate=sampling_rate, 
        return_tensors="pt", 
        padding=True
    )
    
    # Move inputs to the same device as the model
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # Extract features without computing gradients
    with torch.no_grad():
        outputs = model(**inputs)
        
    # outputs.last_hidden_state has shape: (batch_size, sequence_length, hidden_size)
    last_hidden_states = outputs.last_hidden_state
    
    # Mean pooling: Average across the sequence length to get a single vector per audio
    # Output shape becomes: (hidden_size,)
    # Mean across time frame
    embedding = last_hidden_states.mean(dim=1).squeeze().cpu().numpy()
    
    return embedding

# 3. Apply the function to your DataFrames
print(f"Extracting embeddings using device: {device}...")

# Assuming your train_df and test_df are already loaded as shown in your prompt
# IMPORTANT: Wav2Vec 2.0 expects audio sampled at exactly 16,000 Hz.
train_df['w2v_embedding'] = train_df['signal_data'].apply(lambda x: get_w2v_embedding(x, sampling_rate=16000))
test_df['w2v_embedding'] = test_df['signal_data'].apply(lambda x: get_w2v_embedding(x, sampling_rate=16000))

print("Embeddings extracted successfully!")
print(test_df.head())

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 46291.36it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting embeddings using device: cpu...
Embeddings extracted successfully!
                                           signal_data language speaker  \
104  [-0.0234375, -0.015625, -0.0078125, -0.0078125...       SI       C   
218  [-0.14410400390625, -0.144775390625, -0.145050...       SA       E   
64   [-0.0078125, -0.0078125, -0.0078125, 0.0, -0.0...       IT       D   
6    [-3.0517578125e-05, 0.0, 0.0, -3.0517578125e-0...       PO       A   
127  [-0.000152587890625, 0.0001220703125, 0.000152...       FR       B   

     label gender  c  length  \
104      8      M  c    7500   
218      9      F  c   11251   
64       9      F  c   24456   
6        7      M  c   10393   
127      4      F  c    7001   

                                         w2v_embedding  
104  [0.172668, 0.16687207, 0.0005224917, 0.2749196...  
218  [0.12205984, 0.061295304, 0.0839988, 0.1493106...  
64   [0.13495167, 0.24337417, -0.120708704, 0.07979...  
6    [0.12713993, 0.2634376, -0.031140177, 0.10486

In [11]:
test_df['w2v_embedding'].iloc[0].shape

(768,)